### Import requirements


In [ ]:
import os
import torch
import torchvision
import torch.nn as nn
import torch.nn.parameter

import torchvision.models as models
import torchvision.models.efficientnet
from torchvision.models.feature_extraction import (
    get_graph_node_names,
    create_feature_extractor,
)

from efficientnet_pytorch import EfficientNet
from efficientnet_pytorch.utils import efficientnet
from torchinfo import summary

from typing import Optional
from collections import OrderedDict

In [2]:
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
batch_size = 16
summary(model, input_size=(16, 3, 224, 224))

Layer (type:depth-idx)                                  Output Shape              Param #
EfficientNet                                            [16, 1000]                --
├─Sequential: 1-1                                       [16, 1280, 7, 7]          --
│    └─Conv2dNormActivation: 2-1                        [16, 32, 112, 112]        --
│    │    └─Conv2d: 3-1                                 [16, 32, 112, 112]        864
│    │    └─BatchNorm2d: 3-2                            [16, 32, 112, 112]        64
│    │    └─SiLU: 3-3                                   [16, 32, 112, 112]        --
│    └─Sequential: 2-2                                  [16, 16, 112, 112]        --
│    │    └─MBConv: 3-4                                 [16, 16, 112, 112]        1,448
│    └─Sequential: 2-3                                  [16, 24, 56, 56]          --
│    │    └─MBConv: 3-5                                 [16, 24, 56, 56]          6,004
│    │    └─MBConv: 3-6                              

### $EfficientNetV_2-small$


In [3]:
weights = (
    models.EfficientNet_V2_S_Weights.DEFAULT
)  # models.EfficientNet_V2_S_Weights.IMAGENET1K_V1

effnetv2s = models.efficientnet_v2_s(weights=weights)

train_nodes = get_graph_node_names(effnetv2s)
train_nodes

(['x',
  'features.0',
  'features.1.0.block.0',
  'features.1.0.stochastic_depth',
  'features.1.0.add',
  'features.1.1.block.0',
  'features.1.1.stochastic_depth',
  'features.1.1.add',
  'features.2.0.block.0',
  'features.2.0.block.1',
  'features.2.1.block.0',
  'features.2.1.block.1',
  'features.2.1.stochastic_depth',
  'features.2.1.add',
  'features.2.2.block.0',
  'features.2.2.block.1',
  'features.2.2.stochastic_depth',
  'features.2.2.add',
  'features.2.3.block.0',
  'features.2.3.block.1',
  'features.2.3.stochastic_depth',
  'features.2.3.add',
  'features.3.0.block.0',
  'features.3.0.block.1',
  'features.3.1.block.0',
  'features.3.1.block.1',
  'features.3.1.stochastic_depth',
  'features.3.1.add',
  'features.3.2.block.0',
  'features.3.2.block.1',
  'features.3.2.stochastic_depth',
  'features.3.2.add',
  'features.3.3.block.0',
  'features.3.3.block.1',
  'features.3.3.stochastic_depth',
  'features.3.3.add',
  'features.4.0.block.0',
  'features.4.0.block.1',
 

In [4]:
effnetv2s.classifier[1] = nn.Linear(in_features=1280, out_features=10572)
effnetv2s.classifier[1]

Linear(in_features=1280, out_features=10572, bias=True)

In [5]:
x = torch.rand(1, 3, 112, 112)

return_nodes = {
    "features.0": "feature0",
    "features.1": "feature1",
    "features.2": "feature2",
    "features.3": "feature3",
    "features.4": "feature4",
    "features.5": "feature5",
    "features.6": "feature6",
    "features.7": "feature7",
    "avgpool": "avgpool2d",
    "flatten": "flat_vec",
    "classifier.0": "class0",
    "classifier.1": "class1",
}

features = create_feature_extractor(effnetv2s, return_nodes=return_nodes)
layer_feature = features(x)

print("feature.0 = ", layer_feature["feature0"].shape)
print("feature.1 = ", layer_feature["feature1"].shape)
print("feature.2 = ", layer_feature["feature2"].shape)
print("feature.3 = ", layer_feature["feature3"].shape)
print("feature.4 = ", layer_feature["feature4"].shape)
print("feature.5 = ", layer_feature["feature5"].shape)
print("feature.6 = ", layer_feature["feature6"].shape)
print("feature.7 = ", layer_feature["feature7"].shape)
print("avgpool = ", layer_feature["avgpool2d"].shape)
print("flatten = ", layer_feature["flat_vec"].shape)
print("classifier.0 = ", layer_feature["class0"].shape)
print("classifier.0 = ", layer_feature["class1"].shape)

print(layer_feature["feature7"].type)

feature.0 =  torch.Size([1, 24, 56, 56])
feature.1 =  torch.Size([1, 24, 56, 56])
feature.2 =  torch.Size([1, 48, 28, 28])
feature.3 =  torch.Size([1, 64, 14, 14])
feature.4 =  torch.Size([1, 128, 7, 7])
feature.5 =  torch.Size([1, 160, 7, 7])
feature.6 =  torch.Size([1, 256, 4, 4])
feature.7 =  torch.Size([1, 1280, 4, 4])
avgpool =  torch.Size([1, 1280, 1, 1])
flatten =  torch.Size([1, 1280])
classifier.0 =  torch.Size([1, 1280])
classifier.0 =  torch.Size([1, 10572])
<built-in method type of Tensor object at 0x782fda0c7520>


In [6]:
# Load the EfficientNetV2S model
model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)

# Print the names and types of all the layers in the model
for i, layer in enumerate(model.features):
    print(i, layer)

# Or, if you only want to print the layer names
for name, _ in model.named_children():
    print(name)

0 Conv2dNormActivation(
  (0): Conv2d(3, 24, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  (2): SiLU(inplace=True)
)
1 Sequential(
  (0): FusedMBConv(
    (block): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU(inplace=True)
      )
    )
    (stochastic_depth): StochasticDepth(p=0.0, mode=row)
  )
  (1): FusedMBConv(
    (block): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU(inplace=True)
      )
    )
    (stochastic_depth): StochasticDepth(p=0.005, mode=row)
  )
)
2 Sequential(
  (0): 

In [7]:
def build_model(pretrained=True, fine_tune=True, num_classes=10572):
    if pretrained:
        print("[INFO]: Loading pre-trained weights")
    else:
        print("[INFO]: Not loading pre-trained weights")

    weights = (
        torchvision.models.EfficientNet_V2_S_Weights.DEFAULT
    )  # or torchvision.models.EfficientNet_V2_S_Weights.IMAGENET1K_V1

    model = torchvision.models.efficientnet_v2_s(weights=weights)

    if fine_tune:
        print("[INFO]: Fine-tuning all layers...")
        for params in model.parameters():
            params.requires_grad = True
    elif not fine_tune:
        print("[INFO]: Freezing hidden layers...")
        for params in model.parameters():
            params.requires_grad = False
    # Change the final classification head.
    model.classifier[1] = nn.Linear(
        in_features=1280,
        out_features=num_classes,
    )
    return model

In [8]:
build_model

<function __main__.build_model(pretrained=True, fine_tune=True, num_classes=10572)>

In [9]:
class EfficientNet_ViT(nn.Module):
    def __init__(self):
        super().__init__()
        weights = (
            models.EfficientNet_V2_S_Weights.DEFAULT
        )  # models.EfficientNet_V2_S_Weights.IMAGENET1K_V1
        self.pt_model = models.efficientnet_v2_s(weights=weights)
        self.trimmed_model = nn.Sequential(*list(self.pt_model.children())[:-2])
        self.pt_model: Optional[EfficientNet] = None
        for child in self.trimmed_model.children():
            for name, param in child.named_parameters():
                print("==" * 60)
                print("Child name", name, "Parameter Gradient", param.requires_grad)

        print("++" * 20)
        print(self.pt_model)
        print("++" * 20)
        print(self.trimmed_model)

    def forward(self, x):
        x = self.trimmed_model(x)
        return x

In [10]:
x = torch.rand(1, 3, 224, 224)
effnet_vit = EfficientNet_ViT()
op = effnet_vit(x)
print(op.shape)

Child name 0.0.weight Parameter Gradient True
Child name 0.1.weight Parameter Gradient True
Child name 0.1.bias Parameter Gradient True
Child name 1.0.block.0.0.weight Parameter Gradient True
Child name 1.0.block.0.1.weight Parameter Gradient True
Child name 1.0.block.0.1.bias Parameter Gradient True
Child name 1.1.block.0.0.weight Parameter Gradient True
Child name 1.1.block.0.1.weight Parameter Gradient True
Child name 1.1.block.0.1.bias Parameter Gradient True
Child name 2.0.block.0.0.weight Parameter Gradient True
Child name 2.0.block.0.1.weight Parameter Gradient True
Child name 2.0.block.0.1.bias Parameter Gradient True
Child name 2.0.block.1.0.weight Parameter Gradient True
Child name 2.0.block.1.1.weight Parameter Gradient True
Child name 2.0.block.1.1.bias Parameter Gradient True
Child name 2.1.block.0.0.weight Parameter Gradient True
Child name 2.1.block.0.1.weight Parameter Gradient True
Child name 2.1.block.0.1.bias Parameter Gradient True
Child name 2.1.block.1.0.weight Pa

In [11]:
x = torch.rand(1, 3, 112, 112)
effnet_vit = EfficientNet_ViT()
op = effnet_vit(x)
print(op.shape)

Child name 0.0.weight Parameter Gradient True
Child name 0.1.weight Parameter Gradient True
Child name 0.1.bias Parameter Gradient True
Child name 1.0.block.0.0.weight Parameter Gradient True
Child name 1.0.block.0.1.weight Parameter Gradient True
Child name 1.0.block.0.1.bias Parameter Gradient True
Child name 1.1.block.0.0.weight Parameter Gradient True
Child name 1.1.block.0.1.weight Parameter Gradient True
Child name 1.1.block.0.1.bias Parameter Gradient True
Child name 2.0.block.0.0.weight Parameter Gradient True
Child name 2.0.block.0.1.weight Parameter Gradient True
Child name 2.0.block.0.1.bias Parameter Gradient True
Child name 2.0.block.1.0.weight Parameter Gradient True
Child name 2.0.block.1.1.weight Parameter Gradient True
Child name 2.0.block.1.1.bias Parameter Gradient True
Child name 2.1.block.0.0.weight Parameter Gradient True
Child name 2.1.block.0.1.weight Parameter Gradient True
Child name 2.1.block.0.1.bias Parameter Gradient True
Child name 2.1.block.1.0.weight Pa

In [12]:
print(op.shape)
print("=" * 60)
# print(effnet_vit)
for n, c in effnetv2s.named_children():
    print(n)

torch.Size([1, 1280, 4, 4])
features
avgpool
classifier


### $EfficientNetV_1-b_0$


In [13]:
model = EfficientNet.from_pretrained("efficientnet-b0")
model

Loaded pretrained weights for efficientnet-b0


EfficientNet(
  (_conv_stem): Conv2dStaticSamePadding(
    3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False
    (static_padding): ZeroPad2d((0, 1, 0, 1))
  )
  (_bn0): BatchNorm2d(32, eps=0.001, momentum=0.010000000000000009, affine=True, track_running_stats=True)
  (_blocks): ModuleList(
    (0): MBConvBlock(
      (_depthwise_conv): Conv2dStaticSamePadding(
        32, 32, kernel_size=(3, 3), stride=[1, 1], groups=32, bias=False
        (static_padding): ZeroPad2d((1, 1, 1, 1))
      )
      (_bn1): BatchNorm2d(32, eps=0.001, momentum=0.010000000000000009, affine=True, track_running_stats=True)
      (_se_reduce): Conv2dStaticSamePadding(
        32, 8, kernel_size=(1, 1), stride=(1, 1)
        (static_padding): Identity()
      )
      (_se_expand): Conv2dStaticSamePadding(
        8, 32, kernel_size=(1, 1), stride=(1, 1)
        (static_padding): Identity()
      )
      (_project_conv): Conv2dStaticSamePadding(
        32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False
    

In [14]:
weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
effnetb0 = torchvision.models.efficientnet_b0(weights=weights)
train_nodes = get_graph_node_names(effnetb0)
train_nodes

(['x',
  'features.0',
  'features.1.0.block.0',
  'features.1.0.block.1',
  'features.1.0.block.2',
  'features.2.0.block.0',
  'features.2.0.block.1',
  'features.2.0.block.2',
  'features.2.0.block.3',
  'features.2.1.block.0',
  'features.2.1.block.1',
  'features.2.1.block.2',
  'features.2.1.block.3',
  'features.2.1.stochastic_depth',
  'features.2.1.add',
  'features.3.0.block.0',
  'features.3.0.block.1',
  'features.3.0.block.2',
  'features.3.0.block.3',
  'features.3.1.block.0',
  'features.3.1.block.1',
  'features.3.1.block.2',
  'features.3.1.block.3',
  'features.3.1.stochastic_depth',
  'features.3.1.add',
  'features.4.0.block.0',
  'features.4.0.block.1',
  'features.4.0.block.2',
  'features.4.0.block.3',
  'features.4.1.block.0',
  'features.4.1.block.1',
  'features.4.1.block.2',
  'features.4.1.block.3',
  'features.4.1.stochastic_depth',
  'features.4.1.add',
  'features.4.2.block.0',
  'features.4.2.block.1',
  'features.4.2.block.2',
  'features.4.2.block.3',


In [15]:
x = torch.rand(1, 3, 112, 112)

return_nodes = {
    "features.0": "feature0",
    "features.1": "feature1",
    "features.2": "feature2",
    "features.3": "feature3",
    "features.4": "feature4",
    "features.5": "feature5",
    "features.6": "feature6",
    "features.7": "feature7",
    "features.8": "feature8",
    "avgpool": "avgpool2d",
    "flatten": "flat_vec",
    "classifier.0": "class0",
    "classifier.1": "class1",
}

features = create_feature_extractor(effnetb0, return_nodes=return_nodes)
layer_feature = features(x)

print("feature.0 = ", layer_feature["feature0"].shape)
print("feature.1 = ", layer_feature["feature1"].shape)
print("feature.2 = ", layer_feature["feature2"].shape)
print("feature.3 = ", layer_feature["feature3"].shape)
print("feature.4 = ", layer_feature["feature4"].shape)
print("feature.5 = ", layer_feature["feature5"].shape)
print("feature.6 = ", layer_feature["feature6"].shape)
print("feature.7 = ", layer_feature["feature7"].shape)
print("feature.8 = ", layer_feature["feature8"].shape)
print("avgpool = ", layer_feature["avgpool2d"].shape)
print("flatten = ", layer_feature["flat_vec"].shape)
print("classifier.0 = ", layer_feature["class0"].shape)
print("classifier.0 = ", layer_feature["class1"].shape)

print(layer_feature["feature8"].type)

feature.0 =  torch.Size([1, 32, 56, 56])
feature.1 =  torch.Size([1, 16, 56, 56])
feature.2 =  torch.Size([1, 24, 28, 28])
feature.3 =  torch.Size([1, 40, 14, 14])
feature.4 =  torch.Size([1, 80, 7, 7])
feature.5 =  torch.Size([1, 112, 7, 7])
feature.6 =  torch.Size([1, 192, 4, 4])
feature.7 =  torch.Size([1, 320, 4, 4])
feature.8 =  torch.Size([1, 1280, 4, 4])
avgpool =  torch.Size([1, 1280, 1, 1])
flatten =  torch.Size([1, 1280])
classifier.0 =  torch.Size([1, 1280])
classifier.0 =  torch.Size([1, 1000])
<built-in method type of Tensor object at 0x782f7c53fbb0>


In [16]:
print(len(model._blocks))
print(model._blocks[15])

16
MBConvBlock(
  (_expand_conv): Conv2dStaticSamePadding(
    192, 1152, kernel_size=(1, 1), stride=(1, 1), bias=False
    (static_padding): Identity()
  )
  (_bn0): BatchNorm2d(1152, eps=0.001, momentum=0.010000000000000009, affine=True, track_running_stats=True)
  (_depthwise_conv): Conv2dStaticSamePadding(
    1152, 1152, kernel_size=(3, 3), stride=[1, 1], groups=1152, bias=False
    (static_padding): ZeroPad2d((1, 1, 1, 1))
  )
  (_bn1): BatchNorm2d(1152, eps=0.001, momentum=0.010000000000000009, affine=True, track_running_stats=True)
  (_se_reduce): Conv2dStaticSamePadding(
    1152, 48, kernel_size=(1, 1), stride=(1, 1)
    (static_padding): Identity()
  )
  (_se_expand): Conv2dStaticSamePadding(
    48, 1152, kernel_size=(1, 1), stride=(1, 1)
    (static_padding): Identity()
  )
  (_project_conv): Conv2dStaticSamePadding(
    1152, 320, kernel_size=(1, 1), stride=(1, 1), bias=False
    (static_padding): Identity()
  )
  (_bn2): BatchNorm2d(320, eps=0.001, momentum=0.01000000000

In [17]:
inputs = torch.rand(1, 3, 112, 112)
endpoints = model.extract_endpoints(inputs)
print(endpoints["reduction_2"].size())
print(endpoints["reduction_3"].size())

torch.Size([1, 24, 28, 28])
torch.Size([1, 40, 14, 14])


### $Trimmed\  EfficientNet + ViT$


In [18]:
blocks_args, global_params = efficientnet(
    width_coefficient=1.0,
    depth_coefficient=1.0,
    image_size=112,
    dropout_rate=0.2,
    drop_connect_rate=0.2,
    num_classes=10572,
    include_top=False,
)

blocks_args

[BlockArgs(num_repeat=1, kernel_size=3, stride=[1], expand_ratio=1, input_filters=32, output_filters=16, se_ratio=0.25, id_skip=True),
 BlockArgs(num_repeat=2, kernel_size=3, stride=[2], expand_ratio=6, input_filters=16, output_filters=24, se_ratio=0.25, id_skip=True),
 BlockArgs(num_repeat=2, kernel_size=5, stride=[2], expand_ratio=6, input_filters=24, output_filters=40, se_ratio=0.25, id_skip=True),
 BlockArgs(num_repeat=3, kernel_size=3, stride=[2], expand_ratio=6, input_filters=40, output_filters=80, se_ratio=0.25, id_skip=True),
 BlockArgs(num_repeat=3, kernel_size=5, stride=[1], expand_ratio=6, input_filters=80, output_filters=112, se_ratio=0.25, id_skip=True),
 BlockArgs(num_repeat=4, kernel_size=5, stride=[2], expand_ratio=6, input_filters=112, output_filters=192, se_ratio=0.25, id_skip=True),
 BlockArgs(num_repeat=1, kernel_size=3, stride=[1], expand_ratio=6, input_filters=192, output_filters=320, se_ratio=0.25, id_skip=True)]

In [19]:
print(blocks_args)
print(global_params)

[BlockArgs(num_repeat=1, kernel_size=3, stride=[1], expand_ratio=1, input_filters=32, output_filters=16, se_ratio=0.25, id_skip=True), BlockArgs(num_repeat=2, kernel_size=3, stride=[2], expand_ratio=6, input_filters=16, output_filters=24, se_ratio=0.25, id_skip=True), BlockArgs(num_repeat=2, kernel_size=5, stride=[2], expand_ratio=6, input_filters=24, output_filters=40, se_ratio=0.25, id_skip=True), BlockArgs(num_repeat=3, kernel_size=3, stride=[2], expand_ratio=6, input_filters=40, output_filters=80, se_ratio=0.25, id_skip=True), BlockArgs(num_repeat=3, kernel_size=5, stride=[1], expand_ratio=6, input_filters=80, output_filters=112, se_ratio=0.25, id_skip=True), BlockArgs(num_repeat=4, kernel_size=5, stride=[2], expand_ratio=6, input_filters=112, output_filters=192, se_ratio=0.25, id_skip=True), BlockArgs(num_repeat=1, kernel_size=3, stride=[1], expand_ratio=6, input_filters=192, output_filters=320, se_ratio=0.25, id_skip=True)]
GlobalParams(width_coefficient=1.0, depth_coefficient=1.

In [20]:
model = EfficientNet(
    blocks_args=blocks_args,
    global_params=global_params,
)

model

EfficientNet(
  (_conv_stem): Conv2dStaticSamePadding(
    3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False
    (static_padding): ZeroPad2d((0, 1, 0, 1))
  )
  (_bn0): BatchNorm2d(32, eps=0.001, momentum=0.010000000000000009, affine=True, track_running_stats=True)
  (_blocks): ModuleList(
    (0): MBConvBlock(
      (_depthwise_conv): Conv2dStaticSamePadding(
        32, 32, kernel_size=(3, 3), stride=[1, 1], groups=32, bias=False
        (static_padding): ZeroPad2d((1, 1, 1, 1))
      )
      (_bn1): BatchNorm2d(32, eps=0.001, momentum=0.010000000000000009, affine=True, track_running_stats=True)
      (_se_reduce): Conv2dStaticSamePadding(
        32, 8, kernel_size=(1, 1), stride=(1, 1)
        (static_padding): Identity()
      )
      (_se_expand): Conv2dStaticSamePadding(
        8, 32, kernel_size=(1, 1), stride=(1, 1)
        (static_padding): Identity()
      )
      (_project_conv): Conv2dStaticSamePadding(
        32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False
    

In [21]:
print(len(model._blocks))
print(list(model._modules.keys()))

16
['_conv_stem', '_bn0', '_blocks', '_conv_head', '_bn1', '_avg_pooling', '_swish']


In [22]:
for n, c in model.named_children():
    print(n)
    if n == "_blocks":
        print(c._modules.keys())

_conv_stem
_bn0
_blocks
dict_keys(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15'])
_conv_head
_bn1
_avg_pooling
_swish


In [23]:
print(model._blocks[15])

MBConvBlock(
  (_expand_conv): Conv2dStaticSamePadding(
    192, 1152, kernel_size=(1, 1), stride=(1, 1), bias=False
    (static_padding): Identity()
  )
  (_bn0): BatchNorm2d(1152, eps=0.001, momentum=0.010000000000000009, affine=True, track_running_stats=True)
  (_depthwise_conv): Conv2dStaticSamePadding(
    1152, 1152, kernel_size=(3, 3), stride=[1, 1], groups=1152, bias=False
    (static_padding): ZeroPad2d((1, 1, 1, 1))
  )
  (_bn1): BatchNorm2d(1152, eps=0.001, momentum=0.010000000000000009, affine=True, track_running_stats=True)
  (_se_reduce): Conv2dStaticSamePadding(
    1152, 48, kernel_size=(1, 1), stride=(1, 1)
    (static_padding): Identity()
  )
  (_se_expand): Conv2dStaticSamePadding(
    48, 1152, kernel_size=(1, 1), stride=(1, 1)
    (static_padding): Identity()
  )
  (_project_conv): Conv2dStaticSamePadding(
    1152, 320, kernel_size=(1, 1), stride=(1, 1), bias=False
    (static_padding): Identity()
  )
  (_bn2): BatchNorm2d(320, eps=0.001, momentum=0.01000000000000

In [24]:
class EfficientNetTrim(nn.Module):
    def __init__(self):
        super().__init__()

        # self.pt_model = EfficientNet.from_pretrained('efficientnet-b0')

        # 210224 1324 create model from scratch without using pretrained weights
        blocks_args, global_params = efficientnet(
            width_coefficient=1.0,
            depth_coefficient=1.0,
            image_size=112,
            dropout_rate=0.2,
            drop_connect_rate=0.2,
            num_classes=10572,
            include_top=False,
        )

        self.pt_model = EfficientNet(
            blocks_args=blocks_args, global_params=global_params
        )

        self.layers: list[str] = list(
            self.pt_model._modules.keys()
        )  # ['_conv_stem', '_bn0', '_blocks', '_conv_head', '_bn1', '_avg_pooling', '_dropout', '_fc', '_swish'] # Convertion from Odict_keys

        self.layer_count: int = 0

        for layer_name in self.layers:
            if layer_name != "_blocks":
                self.layer_count += 1
            else:
                self.pt_model._blocks = nn.Sequential(
                    *[self.pt_model._blocks[i] for i in range(4)]
                )
                break

        print("Layer Count:", self.layer_count)
        print("Length of _blocks:", len(self.pt_model._blocks))

        for i in range(1, len(self.layers) - self.layer_count):
            layer_name = self.layers[-i]
            popped = self.pt_model._modules.pop(layer_name)
            print("popped Layer:", layer_name, popped)

        # self.pt_model_trim = nn.Sequential(*list(self.pt_model._modules))
        self.pt_model_trim = nn.Sequential(OrderedDict(self.pt_model._modules))
        self.pt_model: Optional[EfficientNet] = None
        self.pt_model: Optional[EfficientNet] = None

    def forward(self, x):
        for n, c in self.pt_model_trim.named_children():
            print(n)  # returns _conv_stem_bn0_blocks
            print(type(c))
            for m, p in c.named_parameters():
                print(m, p.requires_grad)  # returns True
        return self.pt_model_trim(x)

In [25]:
# new_model = EfficientNetTrim()
# summary(new_model,input_size=(3, 224, 224))

x = torch.rand(1, 3, 112, 112)
efficientnet_trim = EfficientNetTrim()
output = efficientnet_trim(x)

Layer Count: 2
Length of _blocks: 4
popped Layer: _swish MemoryEfficientSwish()
popped Layer: _avg_pooling AdaptiveAvgPool2d(output_size=1)
popped Layer: _bn1 BatchNorm2d(1280, eps=0.001, momentum=0.010000000000000009, affine=True, track_running_stats=True)
popped Layer: _conv_head Conv2dStaticSamePadding(
  320, 1280, kernel_size=(1, 1), stride=(1, 1), bias=False
  (static_padding): Identity()
)
_conv_stem
<class 'efficientnet_pytorch.utils.Conv2dStaticSamePadding'>
weight True
_bn0
<class 'torch.nn.modules.batchnorm.BatchNorm2d'>
weight True
bias True
_blocks
<class 'torch.nn.modules.container.Sequential'>
0._depthwise_conv.weight True
0._bn1.weight True
0._bn1.bias True
0._se_reduce.weight True
0._se_reduce.bias True
0._se_expand.weight True
0._se_expand.bias True
0._project_conv.weight True
0._bn2.weight True
0._bn2.bias True
1._expand_conv.weight True
1._bn0.weight True
1._bn0.bias True
1._depthwise_conv.weight True
1._bn1.weight True
1._bn1.bias True
1._se_reduce.weight True
1._s

In [26]:
print(output.shape)

torch.Size([1, 40, 14, 14])


In [ ]:
model = torch.load(
    "./results/EfficientNet_Trim_ViT_casia_cosface_s1/Backbone_EffNet_trim_VIT_checkpoint.pth"
)
print(model)

In [ ]:
PATH = "./results/EfficientNet_Trim_ViT_casia_cosface_s1/Backbone_EffNet_trim_VIT_checkpoint.pth"
model = torch.load(
    "./results/EfficientNet_Trim_ViT_casia_cosface_s1/Backbone_EffNet_trim_VIT_checkpoint.pth"
)
# state = {
#     "model_state_dict": model.state_dict(),
# }
# torch.save(state, PATH)
model.load_state_dict(torch.load(PATH)["model"])
# print weights
for key, val in model.named_parameters():
    print(key, val)

In [27]:
DATA_ROOT = "./data/casia-webface/"
INPUT_SIZE = [112, 112]
with open(os.path.join(DATA_ROOT, "property"), "r") as f:
    NUM_CLASS, h, w = [int(i) for i in f.read().split(",")]
assert h == INPUT_SIZE[0] and w == INPUT_SIZE[1]
print("Number of Training Classes: {}".format(NUM_CLASS))

Number of Training Classes: 10572
